# CTA Top-30 vs CTA Top-30 + Whale Top-20

This notebook has two goals:

1. On the **common sample covered by whale data**, compare the after-cost equity curves of `CTA Top-30` against `CTA Top-30 + Whale Top-20`, directly answering "do whale signals augment traditional CTA signals?"
2. In the same notebook, reproduce the **WhaleSignal Test** from the screenshot: first use 30 traditional CTA signals to explain next-bar returns, then use 20 whale signals to explain the residuals left over from stage 1.

Notes:
- For a fair comparison, both CTA and Whale are evaluated only within the valid whale-coverage window of `cleaned_1m_with_whale.csv`.
- The whale residual test still follows the two-stage logic from the screenshot, but replaces the screenshot's 6 whale signals with the lab's current **Top-20 whale signals**.

In [1]:

from __future__ import annotations

from pathlib import Path
import os
import re
import sys
import warnings

ROOT = Path.cwd()
if not (ROOT / 'tool').exists() and (ROOT.parent / 'tool').exists():
    ROOT = ROOT.parent

MPL_DIR = ROOT / 'var' / 'mplconfig'
MPL_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPL_DIR.resolve()))

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import Markdown, display

from tool.cleaned_table import load_cleaned_minute_whale_csv
from tool.core_cta_baseline import build_core_signal_library
from tool.cta_catalog_execution import (
    ensemble_position_equal_weight,
    equity_curve_from_1m_net,
    net_return_with_costs,
    positions_table_from_catalog,
    split_signal_inv,
    upsample_position_to_1m,
)
from tool.cta_multi_freq_lab import prepare_cta_feature_frame, resample_ohlcv_1m
from tool.cta_signal_catalog import load_catalog
from tool.cta_signal_lab import BacktestConfig, compute_strategy_metrics, flatten_signal_library
from tool.newmath import apply_formulas, numeric_to_float32
from tool.whale_signal_lab import (
    CFG_BASE_DEFAULT,
    build_position_matrix_1m,
    build_raw_signal_matrix_1m,
    build_whale_alpha_library,
    fill_whale_missing,
    first_oos_time_utc,
    list_whale_numeric_columns,
    run_whale_signal_sweep,
    train_test_masks_whale_last_pct,
    trim_whale_window,
)

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.style.use('seaborn-v0_8-whitegrid')
pd.options.display.max_colwidth = 120

from tool.cta_strategy_diagnostics import position_management_stats


In [ ]:

TOP_CTA = 30
TOP_WHALE = 20
TEST_FRAC = 0.30
PRIMARY_MODEL = 'CTA30 + Whale20 (50 signals EW)'
ROBUST_MODEL = 'CTA / Whale 50-50 sleeves'
FEE_BPS = 7.0

CFG_WHALE_EXEC = BacktestConfig(
    freq=CFG_BASE_DEFAULT.freq,
    fee_bps=FEE_BPS,
    slippage_bps=CFG_BASE_DEFAULT.slippage_bps,
    signal_shift=CFG_BASE_DEFAULT.signal_shift,
    signal_clip=CFG_BASE_DEFAULT.signal_clip,
    max_abs_position=CFG_BASE_DEFAULT.max_abs_position,
    position_step=CFG_BASE_DEFAULT.position_step,
)

CFG_1M_EXEC = BacktestConfig(
    freq='1m',
    fee_bps=FEE_BPS,
    slippage_bps=CFG_BASE_DEFAULT.slippage_bps,
    signal_shift=0,
    signal_clip=CFG_BASE_DEFAULT.signal_clip,
    max_abs_position=CFG_BASE_DEFAULT.max_abs_position,
    position_step=CFG_BASE_DEFAULT.position_step,
)

df1 = load_cleaned_minute_whale_csv(ROOT / 'data' / 'cleaned' / 'cleaned_1m_with_whale.csv', root_dir=ROOT)
whale_base_cols = list_whale_numeric_columns(df1)
df1 = trim_whale_window(df1, whale_base_cols)
fill_whale_missing(df1, whale_base_cols, policy='zero')
df1 = prepare_cta_feature_frame(df1)
df1 = numeric_to_float32(df1, exclude=('time_utc',))

ret_1 = df1['ret_1'].astype('float64')
train_mask, test_mask = train_test_masks_whale_last_pct(df1, test_frac=TEST_FRAC)
split_t = first_oos_time_utc(df1)

sample_summary = pd.DataFrame([
    {
        'rows': len(df1),
        'start_utc': df1['time_utc'].iloc[0],
        'end_utc': df1['time_utc'].iloc[-1],
        'whale_numeric_cols': len(whale_base_cols),
        'train_rows': int(train_mask.sum()),
        'test_rows': int(test_mask.sum()),
        'first_oos_bar': split_t,
    }
])
display(sample_summary)


## 1. Signal pool reconstruction

- `CTA Top-30` is loaded directly from `data/cta_signal_catalog_top30.csv`.
- `Whale Top-20` is regenerated using the same `run_whale_signal_sweep` logic as `Whale_Single_Signal_Lab`, then the current Top-20 is taken.

In [ ]:
cta_catalog = load_catalog().head(TOP_CTA).copy()
whale_signal_library = build_whale_alpha_library(whale_base_cols)
whale_sweep = run_whale_signal_sweep(df1, whale_signal_library, cfg_base=CFG_WHALE_EXEC)
whale_top20 = whale_sweep.head(TOP_WHALE).copy()

display(cta_catalog[['signal', 'bar_minutes', 'freq', 'composite', 'train_sharpe', 'test_sharpe']])
display(whale_top20[['signal', 'freq', 'composite_is', 'train_sharpe', 'test_sharpe', 'is_trigger_count', 'os_trigger_count']])

## 2. Equity comparison on the common sample

The main question here is simple: on the **same whale-available sample**, does `CTA Top-30 + Whale Top-20` outperform `CTA Top-30`?

We also keep a `CTA / Whale 50-50 sleeves` model as a robustness check: form the CTA sleeve and Whale sleeve separately, then combine them 50/50, so the result is not dominated by the 30 vs 20 signal-count imbalance.

In [ ]:
cta_pos = positions_table_from_catalog(df1, cta_catalog, top_n=TOP_CTA)
whale_pos = build_position_matrix_1m(whale_top20, df1, whale_signal_library, CFG_WHALE_EXEC)

model_positions = {
    'CTA Top-30': ensemble_position_equal_weight(cta_pos),
    'Whale Top-20': whale_pos.mean(axis=1).clip(-1.0, 1.0),
    PRIMARY_MODEL: pd.concat([cta_pos, whale_pos], axis=1).mean(axis=1).clip(-1.0, 1.0),
    ROBUST_MODEL: (0.5 * ensemble_position_equal_weight(cta_pos) + 0.5 * whale_pos.mean(axis=1)).clip(-1.0, 1.0),
}

model_outputs = {}
metrics_rows = []
risk_rows = []
for model_name, pos in model_positions.items():
    net = net_return_with_costs(pos, ret_1, CFG_1M_EXEC)
    eq = equity_curve_from_1m_net(net)
    model_outputs[model_name] = {'pos': pos, 'net': net, 'eq': eq}

    pm = position_management_stats(pos, CFG_1M_EXEC)
    dd = eq / eq.cummax() - 1.0
    is_underwater = dd < 0
    regime_id = (is_underwater != is_underwater.shift(fill_value=False)).cumsum()
    dd_run_lengths = is_underwater.groupby(regime_id).sum()
    longest_dd_bars = int(dd_run_lengths.max()) if not dd_run_lengths.empty else 0

    net_dt = net.copy()
    net_dt.index = pd.to_datetime(df1.loc[net.index, 'time_utc'], utc=True)
    daily_net = net_dt.resample('1D').sum(min_count=1).dropna()

    risk_rows.append({
        'model': model_name,
        'max_dd': float(dd.min()),
        'longest_dd_days': longest_dd_bars / 1440.0,
        'dd_trough_utc': pd.to_datetime(df1.loc[dd.idxmin(), 'time_utc'], utc=True),
        'worst_1m_net': float(net.min()),
        'worst_1d_net': float(daily_net.min()) if not daily_net.empty else np.nan,
        'mean_abs_pos': pm.mean_abs_pos,
        'pct_time_long': pm.pct_time_long,
        'pct_time_short': pm.pct_time_short,
        'pct_time_flat': pm.pct_time_flat,
        'n_position_changes': pm.n_position_changes,
        'avg_days_between_changes': pm.avg_bars_between_changes / 1440.0,
    })

    for segment, mask in {
        'full': pd.Series(True, index=pos.index),
        'is': train_mask.reindex(pos.index).fillna(False),
        'oos': test_mask.reindex(pos.index).fillna(False),
    }.items():
        m = compute_strategy_metrics(net.loc[mask], CFG_1M_EXEC, position=pos.loc[mask])
        metrics_rows.append({
            'model': model_name,
            'segment': segment,
            'ann_ret': m.get('ann_ret', np.nan),
            'ann_vol': m.get('ann_vol', np.nan),
            'sharpe': m.get('sharpe', np.nan),
            'calmar': m.get('calmar', np.nan),
            'max_dd': m.get('max_dd', np.nan),
            'turnover': m.get('turnover', np.nan),
            'hit_rate': m.get('hit_rate', np.nan),
            'profit_factor': m.get('profit_factor', np.nan),
            'obs': m.get('obs', np.nan),
        })

metrics_df = pd.DataFrame(metrics_rows)
risk_df = pd.DataFrame(risk_rows)

highlight_rows = {
    ('CTA Top-30', 'oos'),
    ('CTA / Whale 50-50 sleeves', 'oos'),
}

def highlight_target_rows(row):
    key = (row['model'], row['segment'])
    if key in highlight_rows:
        return ['background-color: #fee2e2; color: #991b1b; font-weight: bold;'] * len(row)
    return [''] * len(row)

display(
    metrics_df.round(4).style.apply(highlight_target_rows, axis=1)
)

display(risk_df.round(4))

base = metrics_df.loc[metrics_df['model'] == 'CTA Top-30'].set_index('segment')
combo = metrics_df.loc[metrics_df['model'] == PRIMARY_MODEL].set_index('segment')
augmentation_df = pd.DataFrame({
    'ann_ret_lift': combo['ann_ret'] - base['ann_ret'],
    'sharpe_lift': combo['sharpe'] - base['sharpe'],
    'max_dd_reduction_abs': base['max_dd'].abs() - combo['max_dd'].abs(),
    'hit_rate_lift': combo['hit_rate'] - base['hit_rate'],
    'profit_factor_lift': combo['profit_factor'] - base['profit_factor'],
    'turnover_delta': combo['turnover'] - base['turnover'],
}).round(4)

display(augmentation_df)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
colors = {'CTA Top-30': '#1d4ed8', PRIMARY_MODEL: '#d97706'}

for name in ['CTA Top-30', PRIMARY_MODEL]:
    eq = model_outputs[name]['eq']
    t = pd.to_datetime(df1.loc[eq.index, 'time_utc'], utc=True)
    axes[0].plot(t, eq.values, lw=1.8, label=name, color=colors[name])

axes[0].axvline(split_t, color='0.2', ls='--', lw=1.1, alpha=0.9, label='IS | OOS')
axes[0].set_title('Full-sample equity curve (costs included)')
axes[0].set_xlabel('Time (UTC)')
axes[0].set_ylabel('Equity (start = 1)')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

for name in ['CTA Top-30', PRIMARY_MODEL]:
    eq = model_outputs[name]['eq']
    t = pd.to_datetime(df1.loc[eq.index, 'time_utc'], utc=True)
    keep = t >= split_t
    eq_oos = eq.loc[keep]
    t_oos = t[keep]
    axes[1].plot(t_oos, (eq_oos / eq_oos.iloc[0]).values, lw=1.8, label=name, color=colors[name])

axes[1].set_title('OOS zoom (rebased to 1 at OOS start)')
axes[1].set_xlabel('Time (UTC)')
axes[1].set_ylabel('OOS equity (rebased)')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 2b. No-Lookahead Audit

The table below makes the most easily-misunderstood points in this notebook explicit. The key distinctions are:

- **Trading equity comparison** uses positions that have already been `shift(1)`-ed, so the current bar's signal is never used to earn the current bar's return.
- **WhaleSignal Test** is a two-stage explanatory regression. The target is defined as `f_{t+1}`, i.e. signals observable at time `t` are used to explain the next-bar return; coefficient estimation and standardization are both done using the training set only, and then evaluated on OOS.

In [ ]:

lookahead_audit = pd.DataFrame([
    {
        'component': 'CTA trading sleeve',
        'data_available_at_t': 'CTA raw signal on bar t',
        'how_future_data_is_blocked': 'position_from_catalog_row applies signal_to_position(...) and then shift(1) before 1m forward-fill',
        'implementation': 'tool/cta_catalog_execution.position_from_catalog_row',
    },
    {
        'component': 'Whale trading sleeve',
        'data_available_at_t': 'Whale raw signal on bar t',
        'how_future_data_is_blocked': 'position_low_freq_for_row applies signal_to_position(...) and then shift(1) before 1m forward-fill',
        'implementation': 'tool/whale_signal_lab.position_low_freq_for_row',
    },
    {
        'component': 'WhaleSignal regression target',
        'data_available_at_t': 'Signal values at time t only',
        'how_future_data_is_blocked': 'Target is next-bar return close[t+1] / close[t] - 1, so regressors at t explain f_{t+1}',
        'implementation': 'Section 3 notebook cell',
    },
    {
        'component': 'OOS model fit / scaling',
        'data_available_at_t': 'Training block only',
        'how_future_data_is_blocked': 'Signal z-scoring uses train-sample mean/std only; OLS fit uses train rows only; OOS rows are held out',
        'implementation': 'Section 3 notebook cell',
    },
    {
        'component': 'Whale Top-20 selection',
        'data_available_at_t': 'In-sample block only',
        'how_future_data_is_blocked': 'run_whale_signal_sweep ranks by composite_is built from train-only metrics and robustness columns',
        'implementation': 'tool/whale_signal_lab.run_whale_signal_sweep',
    },
])
display(lookahead_audit)


## 3. WhaleSignal Test

A two-stage test following the screenshot's logic:

- **Stage 1**: `f_{t+1} = alpha + sum beta_i * CTA_signal_{i,t} + e_{t+1}`
- **Stage 2**: `e_{t+1} = gamma_0 + sum gamma_j * Whale_signal_{j,t} + epsilon_{t+1}`

To stay close to the screenshot's notion of "signal", this section uses **raw formula signals** rather than post-trading positions. All low-frequency signals are first computed on their native bar frequency, then forward-filled onto the 1m timeline. Coefficient estimation is done only on the IS window; OOS incremental prediction metrics are also reported.

In [ ]:

def build_cta_raw_signal_matrix_1m(df_1m: pd.DataFrame, catalog: pd.DataFrame) -> pd.DataFrame:
    flat = flatten_signal_library(build_core_signal_library())
    frame_cache = {}
    cols = {}
    for j, (_, row) in enumerate(catalog.iterrows()):
        sig = str(row['signal'])
        base_sig, inv = split_signal_inv(sig)
        bar_minutes = int(row['bar_minutes'])
        if bar_minutes not in frame_cache:
            dfr = df_1m.copy() if bar_minutes <= 1 else resample_ohlcv_1m(df_1m, bar_minutes)
            frame_cache[bar_minutes] = prepare_cta_feature_frame(dfr)
        dfr = frame_cache[bar_minutes].copy()
        dfr = apply_formulas(dfr, {base_sig: flat[base_sig]})
        raw = pd.to_numeric(dfr[base_sig], errors='coerce').astype('float64')
        if inv:
            raw = -raw
        cols[f"{j:02d}_{sig[:48]}|{row.get('freq', bar_minutes)}"] = upsample_position_to_1m(raw, df_1m.index)
    return pd.DataFrame(cols, index=df_1m.index, dtype='float64')

cta_raw_1m = build_cta_raw_signal_matrix_1m(df1, cta_catalog)
whale_raw_1m = build_raw_signal_matrix_1m(whale_top20, df1, whale_signal_library)

cta_mu = cta_raw_1m.loc[train_mask].mean()
cta_sd = cta_raw_1m.loc[train_mask].std(ddof=0).replace(0.0, np.nan)
whale_mu = whale_raw_1m.loc[train_mask].mean()
whale_sd = whale_raw_1m.loc[train_mask].std(ddof=0).replace(0.0, np.nan)

cta_z = ((cta_raw_1m - cta_mu) / cta_sd).clip(-6, 6).fillna(0.0)
whale_z = ((whale_raw_1m - whale_mu) / whale_sd).clip(-6, 6).fillna(0.0)
y = (df1['close'].shift(-1) / df1['close'] - 1.0).astype('float64')

keep = y.notna() & cta_z.notna().all(axis=1) & whale_z.notna().all(axis=1)
y = y.loc[keep]
cta_z = cta_z.loc[keep]
whale_z = whale_z.loc[keep]
train_mask_reg = train_mask.loc[keep]
test_mask_reg = test_mask.loc[keep]

X_cta_train = sm.add_constant(cta_z.loc[train_mask_reg], has_constant='add')
X_joint_train = sm.add_constant(pd.concat([cta_z.loc[train_mask_reg], whale_z.loc[train_mask_reg]], axis=1), has_constant='add')
X_whale_train = sm.add_constant(whale_z.loc[train_mask_reg], has_constant='add')
y_train = y.loc[train_mask_reg]

stage1 = sm.OLS(y_train, X_cta_train).fit()
joint = sm.OLS(y_train, X_joint_train).fit()
stage2 = sm.OLS(stage1.resid, X_whale_train).fit()
f_stat, f_pvalue, _ = joint.compare_f_test(stage1)

X_cta_test = sm.add_constant(cta_z.loc[test_mask_reg], has_constant='add')
X_joint_test = sm.add_constant(pd.concat([cta_z.loc[test_mask_reg], whale_z.loc[test_mask_reg]], axis=1), has_constant='add')
y_test = y.loc[test_mask_reg]
pred_stage1 = stage1.predict(X_cta_test)
pred_joint = joint.predict(X_joint_test)

sst_test = float(((y_test - y_test.mean()) ** 2).sum())
oos_r2_stage1 = 1.0 - float(((y_test - pred_stage1) ** 2).sum()) / sst_test
oos_r2_joint = 1.0 - float(((y_test - pred_joint) ** 2).sum()) / sst_test

whale_test_summary = pd.DataFrame([
    {'metric': 'train_r2_stage1_cta_only', 'value': stage1.rsquared},
    {'metric': 'train_r2_joint_cta_plus_whale', 'value': joint.rsquared},
    {'metric': 'train_incremental_r2_from_whales', 'value': joint.rsquared - stage1.rsquared},
    {'metric': 'train_joint_f_stat_for_whale_block', 'value': f_stat},
    {'metric': 'train_joint_f_pvalue_for_whale_block', 'value': f_pvalue},
    {'metric': 'stage2_residual_r2_whales_only', 'value': stage2.rsquared},
    {'metric': 'oos_r2_stage1_cta_only', 'value': oos_r2_stage1},
    {'metric': 'oos_r2_joint_cta_plus_whale', 'value': oos_r2_joint},
    {'metric': 'oos_incremental_r2_from_whales', 'value': oos_r2_joint - oos_r2_stage1},
    {'metric': 'oos_corr_stage1_cta_only', 'value': np.corrcoef(y_test, pred_stage1)[0, 1]},
    {'metric': 'oos_corr_joint_cta_plus_whale', 'value': np.corrcoef(y_test, pred_joint)[0, 1]},
])

alpha = 0.05

whale_test_coefs = (
    pd.DataFrame({
        'signal': stage2.params.index[1:],
        'coef_stage2': stage2.params.iloc[1:].to_numpy(),
        't_stage2': stage2.tvalues.iloc[1:].to_numpy(),
        'p_stage2': stage2.pvalues.iloc[1:].to_numpy(),
    })
    .sort_values('p_stage2', ascending=True, kind='mergesort')
    .reset_index(drop=True)
)

display(whale_test_summary.round(6))
display(whale_test_coefs.head(10).round(6))

top10 = whale_test_coefs.head(10).copy()
top10['is_sig'] = top10['p_stage2'] < alpha

# Reverse only for plotting so the smallest p-value is at the top.
plot_df = top10.iloc[::-1].copy()

# Significant bars in red, others in gray.
colors = np.where(plot_df['is_sig'], '#dc2626', '#9ca3af')

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(plot_df['signal'], plot_df['coef_stage2'], color=colors)

# Add p-values on bars.
for bar, pval in zip(bars, plot_df['p_stage2']):
    x = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2
    dx = 0.00002 if x >= 0 else -0.00002
    ax.text(
        x + dx,
        y,
        f"p={pval:.4f}",
        va='center',
        ha='left' if x >= 0 else 'right',
        fontsize=9,
    )

# Horizontal split line between significant and non-significant signals.
n_sig = int(top10['is_sig'].sum())
if 0 < n_sig < len(top10):
    split_y = len(top10) - n_sig - 0.5
    ax.axhline(split_y, color='red', linestyle='--', linewidth=1.5, label=f'p = {alpha:.2f} cutoff')

# Make significant y-axis labels red too.
for tick_label, is_sig in zip(ax.get_yticklabels(), plot_df['is_sig']):
    if is_sig:
        tick_label.set_color('#dc2626')
        tick_label.set_fontweight('bold')

ax.set_title('Top Stage-2 whale coefficients (sorted by p-value)')
ax.set_xlabel('Stage-2 coefficient on residual return')
ax.set_ylabel('Whale signal')
ax.grid(True, axis='x', alpha=0.3)
ax.legend(loc='best')
plt.tight_layout()
plt.show()



In [ ]:
base_oos = metrics_df.loc[(metrics_df['model'] == 'CTA Top-30') & (metrics_df['segment'] == 'oos')].iloc[0]
combo_oos = metrics_df.loc[(metrics_df['model'] == PRIMARY_MODEL) & (metrics_df['segment'] == 'oos')].iloc[0]
f_p = float(whale_test_summary.loc[whale_test_summary['metric'] == 'train_joint_f_pvalue_for_whale_block', 'value'].iloc[0])
oos_delta_r2 = float(whale_test_summary.loc[whale_test_summary['metric'] == 'oos_incremental_r2_from_whales', 'value'].iloc[0])

takeaway = '\n'.join([
    '**Takeaway**',
    '',
    f"- On the common whale-coverage sample, the augmented model `CTA Top-30 + Whale Top-20` has an OOS Sharpe of **{combo_oos['sharpe']:.2f}**, higher than the `CTA Top-30` baseline at **{base_oos['sharpe']:.2f}**.",
    f"- Under the same setup, the augmented model's OOS annualized return is **{combo_oos['ann_ret']:.2%}**, versus **{base_oos['ann_ret']:.2%}** for the baseline.",
    f"- The WhaleSignal Test joint F-test p-value is **{f_p:.4f}**, indicating that the whale signal block has statistically incremental explanatory power for residual returns beyond the traditional CTA signals.",
    f"- For OOS prediction, the incremental R^2 from adding whale signals is **{oos_delta_r2:.4f}**. 1m returns are still very hard to predict, but the whale block brings the model closer to the explainable residual than the pure CTA block alone.",
])
display(Markdown(takeaway))